# ICARE telescope-resource corpus — exploratory data analysis (A)

This notebook measures the four flattened ICARE tables in
`data/interim/telescopes/<capture-id>/`: what each resource actually
contains, how complete and internally consistent it is, how the four
resources relate to each other, and which structural/data-quality
questions the decisions stage will need to resolve.

It is **descriptive only**. It normalizes nothing, joins nothing into a
new persisted table, and writes no data file. Real-time operational
availability, weather and queue state are out of scope (Stage 1). GRANDMA
is out of scope. Candidate questions for `B_decisions.ipynb` are collected
as they are found and listed together near the end; a short synthesis
closes the notebook.


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/telescopes").is_dir())
CAPTURE_ID = "capture_20260808_071334"
INTERIM_DIR = ROOT / "data/interim/telescopes" / CAPTURE_ID
RAW_CAPTURE_DIR = ROOT / "data/raw/telescopes/icare" / CAPTURE_ID

TABLE_NAMES = ["telescopes", "instruments", "allocations", "observations"]
EXPECTED_ROWS = {"telescopes": 89, "instruments": 95, "allocations": 38, "observations": 93}
PROVENANCE_COLUMNS = ["raw_source", "capture_id", "raw_file", "page_number"]

TABLES = {}
for table_name in TABLE_NAMES:
    table_path = INTERIM_DIR / f"{table_name}.parquet"
    frame = pd.read_parquet(table_path)
    if len(frame) == 0:  # zero rows is a stop condition, never a result
        raise ValueError(f"Table '{table_name}' loaded 0 rows from {table_path}")
    TABLES[table_name] = frame
    print(f"{table_name:14s} rows={len(frame):4d} columns={frame.shape[1]:3d}  <- {table_path}")

print(f"\ncapture identifier : {CAPTURE_ID}")
print(f"raw capture dir    : {RAW_CAPTURE_DIR}")
print(f"interim dir        : {INTERIM_DIR}")

EMPTY_TOKENS = {"", "[]", "{}"}


def has_content(series):
    """True where a cell holds real content: '', '[]', '{}', null and NaN are empty."""
    def alive(value):
        if value is None:
            return False
        if isinstance(value, str):
            return value.strip() not in EMPTY_TOKENS
        if isinstance(value, float) and np.isnan(value):
            return False
        return True
    return series.map(alive)


def clip(value, limit=60):
    """Keep any displayed value short; long values are truncated on sight."""
    text = str(value)
    return f"{text[:limit]}..." if len(text) > limit else text


def fmt(n):
    """Thousands-separated display for counts that can exceed 999."""
    return f"{n:,}"


def is_json_list_series(series, content):
    """True when every present value is a Stage-2 JSON-list-serialized string."""
    present = series[content]
    return len(present) > 0 and present.map(lambda v: isinstance(v, str) and v.startswith("[")).all()


def summarise(series, content):
    """Describe one column under the rules fixed for this notebook."""
    if not content.any():
        return "EMPTY"
    present = series[content]
    if pd.api.types.is_bool_dtype(series):
        return " | ".join(f"{k}={v}" for k, v in present.value_counts().items())
    if pd.api.types.is_numeric_dtype(series):
        return f"min={present.min():.6g} / median={present.median():.6g} / max={present.max():.6g}"
    text = present.astype(str)
    if is_json_list_series(series, content):
        lengths = text.map(lambda v: len(json.loads(v)))
        return f"JSON list, length min={lengths.min()} / median={lengths.median():.0f} / max={lengths.max()}"
    if text.nunique() <= 12:
        return " | ".join(f"{clip(k)}={v}" for k, v in text.value_counts().items())
    examples = " | ".join(clip(v) for v in text.drop_duplicates().head(3))
    return f"{examples} (+{text.nunique() - 3} more distinct)"


def census(table_name):
    """One row per column: dtype, coverage and a rule-based summary, sparsest first."""
    frame = TABLES[table_name]
    rows = []
    for column in frame.columns:
        series = frame[column]
        content = has_content(series)
        rows.append({
            "column": column,
            "dtype": str(series.dtype),
            "non_null": int(series.notna().sum()),
            "coverage_pct": round(100 * content.sum() / len(frame), 1),
            "distinct": int(series[content].astype(str).nunique()),
            "summary": summarise(series, content),
        })
    return pd.DataFrame(rows).sort_values(["coverage_pct", "column"]).reset_index(drop=True)


def census_footer(table_name, report):
    """One line stating how much of the table carries no usable content."""
    empty = sorted(report.loc[report["coverage_pct"] == 0, "column"])
    low = int((report["coverage_pct"] < 5).sum())
    return (f"{table_name}: entirely empty columns = {len(empty)} | columns under 5% content "
            f"coverage (empty included) = {low} | empty columns = {empty}")


# Findings collected progressively across the notebook, rendered near the end.
QUALITY_FINDINGS = []
DECISION_QUESTIONS = []


def note_finding(table, column, observation, scope):
    QUALITY_FINDINGS.append({"table": table, "column": column, "observation": observation,
                             "measured_scope": scope})


def note_question(question, evidence, fields, scope):
    DECISION_QUESTIONS.append({"question": question, "evidence": evidence,
                               "relevant_table_fields": fields, "measured_scope": scope})


controls = pd.DataFrame(
    [{"table": n, "expected_rows": EXPECTED_ROWS[n], "observed_rows": TABLES[n].shape[0],
      "observed_columns": TABLES[n].shape[1],
      "provenance_columns_present": [c for c in PROVENANCE_COLUMNS if c in TABLES[n].columns],
      "status": "PASS" if TABLES[n].shape[0] == EXPECTED_ROWS[n] else "FAIL"} for n in TABLE_NAMES])
print("\nCONTROLS — interim row counts vs the approved Stage-2 capture")
print(controls.to_string(index=False))


telescopes     rows=  89 columns= 21  <- /home/meneses/project_astronomical/MAFORAI/data/interim/telescopes/capture_20260808_071334/telescopes.parquet
instruments    rows=  95 columns= 42  <- /home/meneses/project_astronomical/MAFORAI/data/interim/telescopes/capture_20260808_071334/instruments.parquet
allocations    rows=  38 columns= 50  <- /home/meneses/project_astronomical/MAFORAI/data/interim/telescopes/capture_20260808_071334/allocations.parquet
observations   rows=  93 columns= 27  <- /home/meneses/project_astronomical/MAFORAI/data/interim/telescopes/capture_20260808_071334/observations.parquet

capture identifier : capture_20260808_071334
raw capture dir    : /home/meneses/project_astronomical/MAFORAI/data/raw/telescopes/icare/capture_20260808_071334
interim dir        : /home/meneses/project_astronomical/MAFORAI/data/interim/telescopes/capture_20260808_071334

CONTROLS — interim row counts vs the approved Stage-2 capture
       table  expected_rows  observed_rows  observed_colu

## Full column census

One row per column for each table: dtype, non-null count, content
coverage (nulls, `''`, `[]`, `{}` all count as no content), distinct-value
count, and a rule-based summary. Sorted sparsest-first so thin columns are
immediately visible.


In [2]:
telescopes_census = census("telescopes")
print(census_footer("telescopes", telescopes_census))
telescopes_census


telescopes: entirely empty columns = 0 | columns under 5% content coverage (empty included) = 1 | empty columns = []


,column,dtype,non_null,coverage_pct,distinct,summary
0,weather_link,object,3,3.4,3,http://www.cleardarksky.com/c/PdrMrtrObMBCkey.html=1 | https://app.weathercloud.net/d5359208089#current=1 | http://clearoutside.com/forecast/-30.47/-70.76?view=midday=1
1,skycam_link,object,7,7.9,6,http://kree.ifa.hawaii.edu/allsky/allsky_last.png=2 | http://132.248.4.16/cgi-bin/viewer/video.jpg=1 | http://bianca.palomar.caltech.edu/images/allsky/AllSkyCurren...=1 | https://algol.palomar.caltech.edu/instruments/allsky/AllSkyC...=1 | https://www.foto-webcam.eu/webcam/wendelstein-west/=1 | https://www.obstech.org/weather/cameras.html=1
2,allocations,object,89,39.3,35,"JSON list, length min=1 / median=1 / max=2"
3,elevation,float64,81,91.0,55,min=36 / median=1870 / max=4215
4,lat,float64,81,91.0,71,min=-75.0998 / median=31.0449 / max=52.4092
5,lon,float64,81,91.0,72,min=-155.672 / median=0.1411 / max=149.082
6,capture_id,object,89,100.0,1,capture_20260808_071334=89
7,created_at,object,89,100.0,89,2022-10-24T08:19:57.772278 | 2023-05-19T14:36:01.754089 | 2023-06-07T15:20:11.223652 (+86 more distinct)
8,diameter,float64,89,100.0,36,min=0.18 / median=0.8 / max=20
9,evening,object,89,100.0,79,"""2026-08-08 19:55:14.411"" | ""2026-08-09 02:33:49.113"" | ""2026-08-08 21:45:32.519"" (+76 more distinct)"


In [3]:
instruments_census = census("instruments")
print(census_footer("instruments", instruments_census))
instruments_census


instruments: entirely empty columns = 5 | columns under 5% content coverage (empty included) = 11 | empty columns = ['across_id', 'last_status_update', 'listener_classname', 'tns_id', 'treasuremap_id']


,column,dtype,non_null,coverage_pct,distinct,summary
0,across_id,object,0,0.0,0,EMPTY
1,last_status_update,object,0,0.0,0,EMPTY
2,listener_classname,object,0,0.0,0,EMPTY
3,tns_id,object,0,0.0,0,EMPTY
4,treasuremap_id,object,0,0.0,0,EMPTY
5,sensitivity_data.ps1::open.exposure_time,float64,2,2.1,2,min=30 / median=105 / max=180
6,sensitivity_data.ps1::open.limiting_magnitude,object,2,2.1,2,19.0=1 | 18=1
7,sensitivity_data.ps1::open.magsys,object,2,2.1,1,ab=2
8,sensitivity_data.ps1::open.zeropoint,float64,2,2.1,2,min=26.3 / median=27.25 / max=28.2
9,configuration_data.specific_configuration.station_name,object,3,3.2,3,Tarot_Chili=1 | Tarot_Calern=1 | Tarot_Reunion=1


In [4]:
allocations_census = census("allocations")
print(census_footer("allocations", allocations_census))
allocations_census


allocations: entirely empty columns = 7 | columns under 5% content coverage (empty included) = 8 | empty columns = ['instrument.across_id', 'instrument.last_status_update', 'instrument.listener_classname', 'instrument.tns_id', 'instrument.treasuremap_id', 'proposal_id', 'validity_ranges']


,column,dtype,non_null,coverage_pct,distinct,summary
0,instrument.across_id,object,0,0.0,0,EMPTY
1,instrument.last_status_update,object,0,0.0,0,EMPTY
2,instrument.listener_classname,object,0,0.0,0,EMPTY
3,instrument.tns_id,object,0,0.0,0,EMPTY
4,instrument.treasuremap_id,object,0,0.0,0,EMPTY
5,proposal_id,object,0,0.0,0,EMPTY
6,validity_ranges,object,4,0.0,0,EMPTY
7,instrument.telescope.weather_link,object,1,2.6,1,https://app.weathercloud.net/d5359208089#current=1
8,instrument.sensitivity_data.ps1::open.exposure_time,float64,2,5.3,2,min=30 / median=105 / max=180
9,instrument.sensitivity_data.ps1::open.limiting_magnitude,object,2,5.3,2,19.0=1 | 18=1


In [5]:
observations_census = census("observations")
print(census_footer("observations", observations_census))
observations_census


observations: entirely empty columns = 5 | columns under 5% content coverage (empty included) = 5 | empty columns = ['airmass', 'field.reference_filter_mags', 'field.reference_filters', 'seeing', 'target_name']


,column,dtype,non_null,coverage_pct,distinct,summary
0,airmass,object,0,0.0,0,EMPTY
1,field.reference_filter_mags,object,0,0.0,0,EMPTY
2,field.reference_filters,object,0,0.0,0,EMPTY
3,seeing,object,0,0.0,0,EMPTY
4,target_name,object,0,0.0,0,EMPTY
5,capture_id,object,93,100.0,1,capture_20260808_071334=93
6,created_at,object,93,100.0,93,2024-09-21T14:09:43.876448 | 2024-06-17T12:53:48.454473 | 2024-06-17T12:53:48.739169 (+90 more distinct)
7,exposure_time,int64,93,100.0,2,min=120 / median=120 / max=180
8,field.created_at,object,93,100.0,47,2024-06-15T16:58:17.343796 | 2024-06-15T17:33:30.782062 | 2024-06-15T17:33:30.928242 (+44 more distinct)
9,field.dec,float64,93,100.0,20,min=-34.5405 / median=27.5243 / max=47.1845


## Primary-key / identity analysis

Each table's own `id` column is inspected for uniqueness, together with
any other column whose name suggests it might be an identifier
(`*_id`, `*_id$`), without assuming in advance which one is the true key.


In [6]:
ID_LIKE_RE = re.compile(r"(^id$|_id$)", re.I)
identity_rows = []

for table_name in TABLE_NAMES:
    frame = TABLES[table_name]
    id_like_columns = [c for c in frame.columns if ID_LIKE_RE.search(c) and not c.startswith(("instrument.", "telescope."))]
    for column in id_like_columns:
        series = frame[column]
        non_null = int(series.notna().sum())
        distinct = int(series.nunique())
        duplicated_mask = series.duplicated(keep=False) & series.notna()
        duplicate_values = sorted(series[duplicated_mask].unique().tolist())[:5]
        identity_rows.append({
            "table": table_name, "candidate_id_column": column,
            "non_null": non_null, "distinct": distinct,
            "duplicate_value_count": int(series[duplicated_mask].shape[0]),
            "example_duplicate_values": duplicate_values,
            "is_unique_over_non_null_rows": non_null == distinct,
        })

identity = pd.DataFrame(identity_rows)
identity


,table,candidate_id_column,non_null,distinct,duplicate_value_count,example_duplicate_values,is_unique_over_non_null_rows
0,telescopes,id,89,89,0,[],True
1,telescopes,capture_id,89,1,89,[capture_20260808_071334],False
2,instruments,id,95,95,0,[],True
3,instruments,telescope_id,95,89,12,"[1, 6, 10, 17, 18]",False
4,instruments,treasuremap_id,0,0,0,[],True
5,instruments,tns_id,0,0,0,[],True
6,instruments,across_id,0,0,0,[],True
7,instruments,capture_id,95,1,95,[capture_20260808_071334],False
8,allocations,proposal_id,0,0,0,[],True
9,allocations,group_id,38,3,37,"[3, 38]",False


In [7]:
print("Reading the table above per resource:")
for table_name in TABLE_NAMES:
    frame = TABLES[table_name]
    id_ok = frame["id"].notna().sum() == frame["id"].nunique() == len(frame)
    print(f"  {table_name:14s}: own 'id' column is a complete, unique primary key = {id_ok} "
          f"({frame['id'].nunique()} distinct of {len(frame)} rows)")

obs = TABLES["observations"]
oid_distinct = obs["observation_id"].nunique()
print(f"\nobservations.observation_id is NOT the primary key: {oid_distinct} distinct values over "
      f"{len(obs)} rows; the actual primary key is 'id'.")
example = obs[obs["observation_id"] == obs["observation_id"].mode().iloc[0]][["id", "observation_id", "instrument_id"]]
print("example of a repeated observation_id value, across different instruments:")
print(example.to_string(index=False))
note_finding("observations", "observation_id", "not a unique identifier despite the name",
             f"{oid_distinct} distinct values over {len(obs)} rows; 'id' is the real primary key")


Reading the table above per resource:
  telescopes    : own 'id' column is a complete, unique primary key = True (89 distinct of 89 rows)
  instruments   : own 'id' column is a complete, unique primary key = True (95 distinct of 95 rows)
  allocations   : own 'id' column is a complete, unique primary key = True (38 distinct of 38 rows)
  observations  : own 'id' column is a complete, unique primary key = True (93 distinct of 93 rows)

observations.observation_id is NOT the primary key: 53 distinct values over 93 rows; the actual primary key is 'id'.
example of a repeated observation_id value, across different instruments:
 id  observation_id  instrument_id
 41               0             23
  1               0             22


## Referential-integrity analysis

`instrument.telescope_id -> telescopes.id`,
`allocation.instrument_id -> instruments.id`, and
`observation.instrument_id -> instruments.id` are the only structured
foreign-key relationships exposed by the flattened columns. Each is
checked directly against the parent table's `id` column.


In [8]:
def check_relation(label, child_table, fk_column, parent_table, parent_id_column="id"):
    child = TABLES[child_table]
    parent_ids = set(TABLES[parent_table][parent_id_column])
    fk = child[fk_column]
    populated = fk.notna()
    referenced = fk[populated]
    present_in_parent = referenced[referenced.isin(parent_ids)]
    absent_from_parent = referenced[~referenced.isin(parent_ids)]
    return {
        "relation": label,
        "child_rows_with_populated_fk": int(populated.sum()),
        "child_rows_with_null_fk": int((~populated).sum()),
        "distinct_referenced_ids": int(referenced.nunique()),
        "referenced_ids_present_in_parent": int(present_in_parent.nunique()),
        "referenced_ids_absent_from_parent": int(absent_from_parent.nunique()),
        "example_absent_ids": sorted(absent_from_parent.unique().tolist())[:5],
    }

relations = pd.DataFrame([
    check_relation("instrument -> telescope", "instruments", "telescope_id", "telescopes"),
    check_relation("allocation -> instrument", "allocations", "instrument_id", "instruments"),
    check_relation("observation -> instrument", "observations", "instrument_id", "instruments"),
])
relations


,relation,child_rows_with_populated_fk,child_rows_with_null_fk,distinct_referenced_ids,referenced_ids_present_in_parent,referenced_ids_absent_from_parent,example_absent_ids
0,instrument -> telescope,95,0,89,89,0,[]
1,allocation -> instrument,38,0,34,34,0,[]
2,observation -> instrument,93,0,3,3,0,[]


In [9]:
print("Coverage of the parent side (not just the child->parent direction):")
tel_ids = set(TABLES["telescopes"]["id"])
inst_ids = set(TABLES["instruments"]["id"])
tel_with_instrument = set(TABLES["instruments"]["telescope_id"])
inst_with_allocation = set(TABLES["allocations"]["instrument_id"])
inst_with_observation = set(TABLES["observations"]["instrument_id"])

print(f"telescopes with at least one instrument : {len(tel_ids & tel_with_instrument)} / {len(tel_ids)}")
print(f"instruments with at least one allocation : {len(inst_ids & inst_with_allocation)} / {len(inst_ids)}")
print(f"instruments with at least one observation: {len(inst_ids & inst_with_observation)} / {len(inst_ids)}")
print(f"distinct instruments actually observed   : {sorted(inst_with_observation)}")
print(f"observed instruments that also carry an allocation: "
      f"{inst_with_observation & inst_with_allocation} of {inst_with_observation}")

note_question(
    "Historical observations only exercise 3 of 95 instruments — can they serve as evidence "
    "of empirical performance for the corpus as a whole, and under what limitations?",
    f"observations.instrument_id covers {len(inst_with_observation)} distinct instrument(s) "
    f"out of {len(inst_ids)} instruments and {len(tel_ids)} telescopes",
    "observations.instrument_id, instruments.id",
    f"{len(TABLES['observations'])} observation rows, {len(inst_with_observation)} instruments",
)


Coverage of the parent side (not just the child->parent direction):
telescopes with at least one instrument : 89 / 89
instruments with at least one allocation : 34 / 95
instruments with at least one observation: 3 / 95
distinct instruments actually observed   : [9, 22, 23]
observed instruments that also carry an allocation: {9, 22, 23} of {9, 22, 23}


## Duplication and structural redundancy

Beyond primary-key duplication (already checked above), Stage 2 preserved
several *nested copies* of the same information: `telescopes.instruments`
and `telescopes.allocations` are serialized lists that duplicate rows of
the standalone `instruments`/`allocations` tables; `instruments.telescope.*`
duplicates `telescopes.*`; `allocations.instrument.*` duplicates
`instruments.*`. Their agreement is measured directly, without deciding
which representation should be canonical.


In [10]:
def whole_row_duplicate_count(table_name, exclude=("raw_file", "page_number")):
    frame = TABLES[table_name]
    compare_columns = [c for c in frame.columns if c not in exclude and c != "id"]
    return int(frame[compare_columns].astype(str).duplicated().sum())

redundancy_rows = []
for table_name in TABLE_NAMES:
    redundancy_rows.append({
        "check": "rows identical on every column except id",
        "table": table_name,
        "finding": f"{whole_row_duplicate_count(table_name)} (0 means no such duplicate exists)",
    })

# instruments.telescope.* vs telescopes.* via telescope_id join
joined = TABLES["instruments"].merge(TABLES["telescopes"], left_on="telescope_id", right_on="id",
                                     suffixes=("_inst", "_tel"))
for field in ["name", "lat", "lon", "diameter", "elevation"]:
    left = joined[f"telescope.{field}"]
    right = joined[field] if field in joined.columns else joined[f"{field}_tel"]
    agree = int(((left == right) | (left.isna() & right.isna())).sum())
    redundancy_rows.append({
        "check": "instruments.telescope.* vs telescopes.* (joined on telescope_id)",
        "table": f"instruments.telescope.{field}",
        "finding": f"agrees on {agree} / {len(joined)} joined rows",
    })

# allocations.instrument.* vs instruments.* via instrument_id join
joined2 = TABLES["allocations"].merge(TABLES["instruments"], left_on="instrument_id", right_on="id",
                                      suffixes=("_alloc", "_inst"))
for field in ["name", "band", "type", "telescope_id"]:
    left = joined2[f"instrument.{field}"]
    right = joined2[field] if field in joined2.columns else joined2[f"{field}_inst"]
    agree = int((left.astype(str) == right.astype(str)).sum())
    redundancy_rows.append({
        "check": "allocations.instrument.* vs instruments.* (joined on instrument_id)",
        "table": f"allocations.instrument.{field}",
        "finding": f"agrees on {agree} / {len(joined2)} joined rows",
    })

redundancy = pd.DataFrame(redundancy_rows)
redundancy


,check,table,finding
0,rows identical on every column except id,telescopes,0 (0 means no such duplicate exists)
1,rows identical on every column except id,instruments,0 (0 means no such duplicate exists)
2,rows identical on every column except id,allocations,0 (0 means no such duplicate exists)
3,rows identical on every column except id,observations,0 (0 means no such duplicate exists)
4,instruments.telescope.* vs telescopes.* (joined on telescope_id),instruments.telescope.name,agrees on 95 / 95 joined rows
5,instruments.telescope.* vs telescopes.* (joined on telescope_id),instruments.telescope.lat,agrees on 95 / 95 joined rows
6,instruments.telescope.* vs telescopes.* (joined on telescope_id),instruments.telescope.lon,agrees on 95 / 95 joined rows
7,instruments.telescope.* vs telescopes.* (joined on telescope_id),instruments.telescope.diameter,agrees on 95 / 95 joined rows
8,instruments.telescope.* vs telescopes.* (joined on telescope_id),instruments.telescope.elevation,agrees on 95 / 95 joined rows
9,allocations.instrument.* vs instruments.* (joined on instrument_id),allocations.instrument.name,agrees on 38 / 38 joined rows


In [11]:
def serialized_id_set(value):
    return sorted(entry["id"] for entry in json.loads(value))

mismatches = []
for _, row in TABLES["telescopes"].iterrows():
    listed_instrument_ids = serialized_id_set(row["instruments"])
    real_instrument_ids = sorted(TABLES["instruments"].loc[TABLES["instruments"]["telescope_id"] == row["id"], "id"])
    if listed_instrument_ids != real_instrument_ids:
        mismatches.append(("telescopes.instruments", row["id"], listed_instrument_ids, real_instrument_ids))

    listed_allocation_ids = serialized_id_set(row["allocations"])
    instrument_ids_for_telescope = set(real_instrument_ids)
    real_allocation_ids = sorted(TABLES["allocations"].loc[
        TABLES["allocations"]["instrument_id"].isin(instrument_ids_for_telescope), "id"])
    if listed_allocation_ids != real_allocation_ids:
        mismatches.append(("telescopes.allocations", row["id"], listed_allocation_ids, real_allocation_ids))

print(f"telescopes.instruments (nested list) vs the standalone instruments table: "
      f"{sum(1 for m in mismatches if m[0] == 'telescopes.instruments')} mismatching telescope(s) of {len(TABLES['telescopes'])}")
print(f"telescopes.allocations (nested list) vs the standalone allocations table (bridged through "
      f"instrument_id): {sum(1 for m in mismatches if m[0] == 'telescopes.allocations')} mismatching "
      f"telescope(s) of {len(TABLES['telescopes'])}")
for check, telescope_id, listed, real in mismatches:
    print(f"  {check}: telescope id={telescope_id} lists {listed}, standalone table has {real}")
    if check == "telescopes.allocations":
        note_finding(
            "telescopes", "allocations (nested)",
            f"telescope id={telescope_id} nests allocation id(s) {sorted(set(listed) - set(real))} "
            f"that do not exist in the standalone allocations table (directly or via its instrument)",
            "1 telescope, 1 nested allocation reference",
        )


telescopes.instruments (nested list) vs the standalone instruments table: 0 mismatching telescope(s) of 89
telescopes.allocations (nested list) vs the standalone allocations table (bridged through instrument_id): 1 mismatching telescope(s) of 89
  telescopes.allocations: telescope id=137 lists [78], standalone table has []


## String and label quality

A generic scan for whitespace, empty strings and the Unicode replacement
character across every text column of all four tables, followed by a
focused case-variant check on the columns most likely to matter for later
normalization: telescope/instrument names, `type`, `band`, `filters` and
`filt`.


In [12]:
generic_findings = []
for table_name in TABLE_NAMES:
    frame = TABLES[table_name]
    for column in frame.columns:
        series = frame[column]
        content = has_content(series)
        if not content.any():
            continue
        present = series[content]
        if not present.map(lambda v: isinstance(v, str)).all():
            continue
        text = present.astype(str)
        stripped = int((text != text.str.strip()).sum())
        if stripped:
            generic_findings.append({"table": table_name, "column": column,
                                     "issue": "leading/trailing whitespace", "affected_cells": stripped})
        replacement = int(text.str.contains("\ufffd", regex=False).sum())
        if replacement:
            generic_findings.append({"table": table_name, "column": column,
                                     "issue": "contains U+FFFD replacement character",
                                     "affected_cells": replacement})
        empty_string = int((series.astype(str).str.strip() == "").sum()) if series.dtype == object else 0

generic = pd.DataFrame(generic_findings)
print(f"columns with a whitespace or replacement-character anomaly: {len(generic)}")
generic if len(generic) else "none found"


columns with a whitespace or replacement-character anomaly: 7


,table,column,issue,affected_cells
0,telescopes,name,leading/trailing whitespace,1
1,instruments,region,leading/trailing whitespace,19
2,instruments,name,leading/trailing whitespace,1
3,instruments,telescope.name,leading/trailing whitespace,1
4,allocations,pi,leading/trailing whitespace,1
5,allocations,instrument.name,leading/trailing whitespace,1
6,allocations,instrument.telescope.name,leading/trailing whitespace,1


In [13]:
def true_case_variants(series):
    """Groups of >=2 genuinely different raw strings sharing one lowercase form."""
    present = series[has_content(series)].astype(str)
    groups = present.groupby(present.str.lower()).agg(lambda x: sorted(set(x)))
    return {k: v for k, v in groups.items() if len(v) > 1}

label_targets = [("telescopes", "name"), ("telescopes", "nickname"),
                 ("instruments", "name"), ("instruments", "band"), ("instruments", "type"),
                 ("instruments", "api_classname"), ("instruments", "api_classname_obsplan"),
                 ("observations", "filt")]
label_rows = []
for table_name, column in label_targets:
    series = TABLES[table_name][column]
    variants = true_case_variants(series)
    affected = int(series[has_content(series)].astype(str).str.lower().isin(variants.keys()).sum())
    label_rows.append({"table": table_name, "column": column,
                       "distinct_values": int(series[has_content(series)].nunique()),
                       "case_variant_groups": len(variants), "affected_cells": affected,
                       "examples": {k: v for k, v in list(variants.items())[:5]}})
labels = pd.DataFrame(label_rows)
labels


,table,column,distinct_values,case_variant_groups,affected_cells,examples
0,telescopes,name,89,0,0,{}
1,telescopes,nickname,89,0,0,{}
2,instruments,name,95,0,0,{}
3,instruments,band,5,1,90,"{'optical': ['Optical', 'optical']}"
4,instruments,type,3,0,0,{}
5,instruments,api_classname,12,0,0,{}
6,instruments,api_classname_obsplan,6,0,0,{}
7,observations,filt,2,0,0,{}


In [14]:
band = TABLES["instruments"]["band"]
print("instruments.band raw value counts:")
print(band.value_counts(dropna=False).to_string())
affected = int(band.astype(str).str.lower().isin(["optical"]).sum())
print(f"\n'Optical'/'optical' case variants affect {affected} of {len(band)} instruments "
      f"({100 * affected / len(band):.1f}%). One additional row combines two bands as free text: "
      f"{band[band.astype(str).str.contains(',', na=False)].tolist()}.")
note_finding("instruments", "band",
             f"{band.nunique()} distinct raw labels, including 'Optical'/'optical' case variants and "
             f"one combined free-text value 'Optical, IR'",
             f"{affected} of {len(band)} instruments carry a case-variant label")
note_question(
    "Do raw `band`/filter labels require a canonical representation before they can be grouped or compared?",
    f"instruments.band has {band.nunique()} distinct raw labels for {band[has_content(band)].shape[0]} "
    f"populated rows, {affected} of which differ only by case",
    "instruments.band, instruments.filters, observations.filt",
    f"{affected} of {len(band)} instruments",
)


instruments.band raw value counts:
band
Optical        68
optical        22
ir              2
Xray            2
Optical, IR     1

'Optical'/'optical' case variants affect 90 of 95 instruments (94.7%). One additional row combines two bands as free text: ['Optical, IR'].


## Telescopes — descriptive profile

Numeric location/size fields, boolean operational-mode flags, and the
mixed bool/string `morning`/`evening` structure already flagged by
Stage 2.


In [15]:
tel = TABLES["telescopes"]
numeric_fields = ["diameter", "lat", "lon", "elevation"]
numeric_rows = []
for field in numeric_fields:
    series = tel[field]
    content = has_content(series)
    present = series[content]
    numeric_rows.append({
        "field": field, "coverage_pct": round(100 * content.sum() / len(tel), 1),
        "non_null": int(content.sum()), "min": present.min(), "median": present.median(),
        "max": present.max(),
    })
pd.DataFrame(numeric_rows)


,field,coverage_pct,non_null,min,median,max
0,diameter,100.0,89,0.18000,0.80000,20.000000
1,lat,91.0,81,-75.09978,31.04487,52.409184
2,lon,91.0,81,-155.67200,0.14110,149.082341
3,elevation,91.0,81,36.00000,1870.00000,4215.000000


In [16]:
print("Boolean / categorical-like telescope fields:")
for field in ["robotic", "fixed_location", "is_night_astronomical"]:
    print(f"  {field:24s}: {tel[field].value_counts().to_dict()}")

no_location = tel[~has_content(tel["lat"])]
print(f"\ntelescopes without a populated lat/lon: {len(no_location)} of {len(tel)}")
print(f"of those, fixed_location == False: {int((no_location['fixed_location'] == False).sum())} "
      f"of {len(no_location)}")

for field in ["weather_link", "skycam_link"]:
    print(f"{field}: populated on {int(has_content(tel[field]).sum())} of {len(tel)} telescopes")

note_finding("telescopes", "lat/lon/elevation",
             f"{len(no_location)} of {len(tel)} telescopes carry no location, all of them non-fixed",
             f"{len(no_location)} rows")


Boolean / categorical-like telescope fields:
  robotic                 : {False: 46, True: 43}
  fixed_location          : {True: 81, False: 8}
  is_night_astronomical   : {False: 63, True: 26}

telescopes without a populated lat/lon: 8 of 89
of those, fixed_location == False: 8 of 8
weather_link: populated on 3 of 89 telescopes
skycam_link: populated on 7 of 89 telescopes


In [17]:
def morning_evening_category(value):
    if value is None:
        return "null"
    if value == "true":
        return "json-bool literal true"
    if value == "false":
        return "json-bool literal false"
    if isinstance(value, str) and value.startswith('"'):
        return "json-quoted string (timestamp-shaped)"
    return f"other: {value!r}"

morning_cat = tel["morning"].map(morning_evening_category)
evening_cat = tel["evening"].map(morning_evening_category)
me = pd.DataFrame({"morning": morning_cat.value_counts(), "evening": evening_cat.value_counts()}).fillna(0).astype(int)
print("morning / evening value categories (Stage-2 mixed-type serialized columns):")
print(me)

quoted = tel.loc[morning_cat.str.startswith("json-quoted"), "morning"].head(3).tolist()
print(f"\nrepresentative quoted values: {quoted}")
same_pattern = (morning_cat.values == evening_cat.values).all()
print(f"morning and evening exhibit the exact same per-row category pattern: {same_pattern}")
print("No 'true' literal was observed in this capture for either column; only 'false' or a quoted "
      "timestamp-shaped string appear.")

note_finding("telescopes", "morning / evening",
             "mixed bool/string values: a literal boolean 'false' on some rows, a quoted "
             "timestamp-shaped string on others; the literal 'true' was never observed",
             f"{int((morning_cat == 'json-bool literal false').sum())} of {len(tel)} rows are the "
             f"boolean form, the rest are the timestamp-shaped string form")
note_question(
    "How should the mixed `morning`/`evening` values (boolean `false` vs. a timestamp-shaped "
    "string) be represented in the final schema?",
    "8 of 89 telescopes carry a literal 'false' in morning/evening while the remaining 81 carry a "
    "quoted timestamp-shaped string; 'true' was never observed",
    "telescopes.morning, telescopes.evening",
    "89 telescopes",
)


morning / evening value categories (Stage-2 mixed-type serialized columns):
                                       morning  evening
json-quoted string (timestamp-shaped)       81       81
json-bool literal false                      8        8

representative quoted values: ['"2026-08-09 00:12:16.047"', '"2026-08-08 10:59:28.596"', '"2026-08-09 01:44:30.662"']
morning and evening exhibit the exact same per-row category pattern: True
No 'true' literal was observed in this capture for either column; only 'false' or a quoted timestamp-shaped string appear.


## Instruments — descriptive profile

Identity, the telescope relationship (already checked structurally
above), and the categorical fields `type`/`band`/`api_classname*`, plus
the presence flags `has_fields`/`has_region`/`number_of_fields`. Filters,
region and sensitivity_data each get a dedicated section below because
they matter enough to characterize on their own.


In [18]:
inst = TABLES["instruments"]
print(f"instruments: {len(inst)} rows, referencing {inst['telescope_id'].nunique()} distinct telescopes")
print("\ntype value counts:")
print(inst["type"].value_counts(dropna=False).to_string())
print("\napi_classname value counts (instrument control-software identity):")
print(inst["api_classname"].value_counts(dropna=False).to_string())
print(f"api_classname coverage: {int(has_content(inst['api_classname']).sum())} / {len(inst)}")
print(f"api_classname_obsplan coverage: {int(has_content(inst['api_classname_obsplan']).sum())} / {len(inst)}")
print(f"last_status_update coverage: {int(has_content(inst['last_status_update']).sum())} / {len(inst)}")
if not has_content(inst["last_status_update"]).any():
    note_finding("instruments", "last_status_update", "entirely empty in this capture", f"0 of {len(inst)}")


instruments: 95 rows, referencing 89 distinct telescopes

type value counts:
type
imager                  81
imaging spectrograph     9
spectrograph             5

api_classname value counts (instrument control-software identity):
api_classname
GENERICAPI    41
None          33
MMAAPI         6
TAROTAPI       4
ATLASAPI       2
TRTAPI         2
ZTFAPI         1
PS1API         1
SEDMAPI        1
UVOTXRTAPI     1
GEMINIAPI      1
TTTAPI         1
IOOAPI         1
api_classname coverage: 62 / 95
api_classname_obsplan coverage: 24 / 95
last_status_update coverage: 0 / 95


In [19]:
print("has_fields / has_region / number_of_fields:")
print(inst[["has_fields", "has_region"]].value_counts())
print(f"\nnumber_of_fields > 0 exactly when has_fields is True: "
      f"{((inst['number_of_fields'] > 0) == inst['has_fields']).all()}")
print("number_of_fields, for instruments where it is > 0:")
print(inst.loc[inst["number_of_fields"] > 0, "number_of_fields"].describe()[["count", "min", "50%", "max"]]
      .map(lambda v: fmt(round(v)) if v == round(v) else v))


has_fields / has_region / number_of_fields:
has_fields  has_region
False       False         78
True        True          12
False       True           5
Name: count, dtype: int64

number_of_fields > 0 exactly when has_fields is True: True
number_of_fields, for instruments where it is > 0:
count         12
min            2
50%       1649.5
max      216,114
Name: number_of_fields, dtype: object


## Filters

`instruments.filters` is a Stage-2 JSON-list column; `observations.filt`
is a direct scalar column. Both are characterized here, then compared to
each other, without building a canonical vocabulary.


In [20]:
filters_series = inst["filters"]
filter_lists = filters_series.map(json.loads)
lengths = filter_lists.map(len)
print(f"instruments.filters coverage: {int(has_content(filters_series).sum())} / {len(inst)} "
      f"(non-null, but {int((lengths == 0).sum())} of those are an empty list)")
print(f"filters per instrument: min={lengths.min()} / median={lengths.median():.0f} / max={lengths.max()}")
print(f"instruments with zero filters: {int((lengths == 0).sum())} of {len(inst)}")

all_filter_labels = sorted({label for one_list in filter_lists for label in one_list})
print(f"\ndistinct raw filter labels across all instruments: {len(all_filter_labels)}")
print(all_filter_labels)

label_frequency = pd.Series([label for one_list in filter_lists for label in one_list]).value_counts()
print(f"\nmost frequent filter labels:\n{label_frequency.head(10).to_string()}")

lower_groups = pd.Series(all_filter_labels).groupby(pd.Series(all_filter_labels).str.lower()).agg(list)
case_variants = {k: v for k, v in lower_groups.items() if len(v) > 1}
print(f"\ncase/whitespace variant groups among filter labels: {case_variants if case_variants else 'none'}")


instruments.filters coverage: 91 / 95 (non-null, but 4 of those are an empty list)
filters per instrument: min=0 / median=6 / max=41
instruments with zero filters: 4 of 95

distinct raw filter labels across all instruments: 56
['2massh', '2massj', '2massks', 'atlasc', 'atlaso', 'bessellb', 'besselli', 'bessellr', 'bessellux', 'bessellv', 'desy', 'epfxt', 'gaia::g', 'gaia::gbp', 'gaia::grp', 'gaia::grvs', 'gotob', 'gotog', 'gotol', 'gotor', 'lsstg', 'lssti', 'lsstr', 'lsstu', 'lssty', 'lsstz', 'ps1::g', 'ps1::i', 'ps1::open', 'ps1::r', 'ps1::w', 'ps1::y', 'ps1::z', 'sdssg', 'sdssi', 'sdssr', 'sdssu', 'sdssz', 'standard::b', 'standard::i', 'standard::r', 'standard::u', 'standard::v', 'svommxt', 'swiftxrt', 'tess', 'uvot::b', 'uvot::u', 'uvot::uvm2', 'uvot::uvw1', 'uvot::uvw2', 'uvot::v', 'uvot::white', 'ztfg', 'ztfi', 'ztfr']

most frequent filter labels:
sdssr        60
sdssi        54
sdssg        53
bessellr     50
bessellb     47
sdssz        46
bessellv     44
besselli     42
ps1::o

In [21]:
print("Relationship between instruments.band and instruments.filters (descriptive only):")
band_filter = pd.DataFrame({"band": inst["band"], "n_filters": lengths})
print(band_filter.groupby("band")["n_filters"].agg(["count", "min", "median", "max"]))

obs_filt = TABLES["observations"]["filt"]
obs_labels = set(obs_filt.dropna())
inst_labels = set(all_filter_labels)
print(f"\nobservation filt labels: {sorted(obs_labels)}")
print(f"observation filt labels also present in some instrument's filters list: {obs_labels & inst_labels}")
print(f"observation filt labels absent from every instrument's filters list: {obs_labels - inst_labels or 'none'}")

# do the instruments that were actually observed list their observed filter?
for instrument_id in sorted(TABLES["observations"]["instrument_id"].unique()):
    used_filters = set(TABLES["observations"].loc[TABLES["observations"]["instrument_id"] == instrument_id, "filt"])
    listed_filters = set(json.loads(inst.loc[inst["id"] == instrument_id, "filters"].iloc[0]))
    print(f"instrument {instrument_id}: observed filt(s) {used_filters}, listed in instruments.filters: "
          f"{used_filters.issubset(listed_filters)}")


Relationship between instruments.band and instruments.filters (descriptive only):
             count  min  median  max
band                                
Optical         68    0     6.0   41
Optical, IR      1    5     5.0    5
Xray             2    1     1.0    1
ir               2    3     5.0    7
optical         22    0     6.0   10

observation filt labels: ['bessellr', 'ps1::open']
observation filt labels also present in some instrument's filters list: {'bessellr', 'ps1::open'}
observation filt labels absent from every instrument's filters list: none
instrument 9: observed filt(s) {'ps1::open'}, listed in instruments.filters: True
instrument 22: observed filt(s) {'bessellr'}, listed in instruments.filters: True
instrument 23: observed filt(s) {'bessellr'}, listed in instruments.filters: True


## Region / footprint information

`region` (a DS9 region-file string), `region_summary`, `has_region`,
`has_fields` and `number_of_fields` are ICARE's own footprint-related
fields. No FoV value is computed here; only what ICARE directly supplies
is measured.


In [22]:
region_present = has_content(inst["region"])
region_summary_present = has_content(inst["region_summary"])
print(f"instruments with a populated 'region' (DS9 text): {int(region_present.sum())} / {len(inst)}")
print(f"instruments with a populated 'region_summary' text: {int(region_summary_present.sum())} / {len(inst)}")
print(f"instruments with has_region == True: {int(inst['has_region'].sum())} / {len(inst)}")
print(f"instruments with NEITHER region text nor region_summary content: "
      f"{int((~region_present & ~region_summary_present).sum())} / {len(inst)}")

crosstab = pd.crosstab(region_present, inst["has_region"],
                       rownames=["region_text_present"], colnames=["has_region_flag"])
print("\nregion text presence vs has_region flag:")
print(crosstab)
mismatch = int(((region_present) & (~inst["has_region"])).sum())
if mismatch:
    print(f"\n{mismatch} instrument(s) carry region text but has_region == False:")
    print(inst.loc[region_present & (~inst["has_region"]), ["id", "name", "has_region"]].to_string(index=False))
    note_finding("instruments", "region / has_region",
                 f"{mismatch} instrument(s) have populated region text while has_region == False",
                 f"{mismatch} of {len(inst)} instruments")


instruments with a populated 'region' (DS9 text): 19 / 95
instruments with a populated 'region_summary' text: 17 / 95
instruments with has_region == True: 17 / 95
instruments with NEITHER region text nor region_summary content: 76 / 95

region text presence vs has_region flag:
has_region_flag      False  True 
region_text_present              
False                   76      0
True                     2     17

2 instrument(s) carry region text but has_region == False:
 id          name  has_region
 26       MMATest       False
 61 Les-Makes/T60       False


In [23]:
print("Representative region_summary values (free text, may embed width/height):")
for value in inst.loc[region_summary_present, "region_summary"].drop_duplicates().head(8):
    print(f"  {value!r}")
print(f"\nrepresentative 'region' DS9 text (first 200 characters of one example):")
print(inst.loc[region_present, "region"].iloc[0][:200])
print("\nNo width/height/unit value is parsed out of this text here; ICARE supplies it as free text, "
      "not as a structured quantity+unit pair.")
note_question(
    "Is ICARE's region/footprint information (DS9 text on 19/95 instruments, a free-text "
    "region_summary on the same 19) sufficiently complete and structured for the intended "
    "telescope-resource layer?",
    f"only {int(region_present.sum())} of {len(inst)} instruments carry any footprint information, "
    f"and it is stored as DS9 region text / free text rather than a structured quantity+unit",
    "instruments.region, instruments.region_summary, instruments.has_region, instruments.number_of_fields",
    f"{int(region_present.sum())} of {len(inst)} instruments",
)


Representative region_summary values (free text, may embed width/height):
  'Rectangle [width, height]: (0.53, 0.53)'
  'Rectangle [width, height]: (1.9, 1.9)'
  'Rectangle [width, height]: (6.8, 6.8)'
  'Rectangle [width, height]: (0.161, 0.449)'
  'Rectangle [width, height]: (0.55, 0.55)'
  'Rectangle [width, height]: (0.4, 0.27)'
  'Rectangle [width, height]: (0.5, 0.5)'
  'Rectangle [width, height]: (0.45, 0.45)'

representative 'region' DS9 text (first 200 characters of one example):
# Region file format: DS9 astropy/regions
global include=1
icrs
polygon(2.79091200,-2.01070000,2.79064100,-2.86593800,3.64472500,-2.86593800,3.64445400,-2.01070000)
polygon(1.92541400,-2.00760400,1.92

No width/height/unit value is parsed out of this text here; ICARE supplies it as free text, not as a structured quantity+unit pair.


## Sensitivity data

The complete `sensitivity_data` family flattened by Stage 2, including the
mixed int/float `limiting_magnitude` already discovered there.


In [24]:
sens_columns = [c for c in inst.columns if c.startswith("sensitivity_data.")]
print(f"sensitivity_data.* columns discovered by Stage 2: {sens_columns}")

sens_rows = []
for column in sens_columns:
    series = inst[column]
    content = has_content(series)
    sens_rows.append({"field": column.split(".", 1)[1], "coverage_pct": round(100 * content.sum() / len(inst), 1),
                      "non_null": int(content.sum())})
print(pd.DataFrame(sens_rows).to_string(index=False))

carrying = inst[inst[sens_columns].apply(has_content).any(axis=1)]
print(f"\ninstruments carrying ANY sensitivity_data content: {len(carrying)} of {len(inst)}")
print(carrying[["id", "name", "band"] + sens_columns].to_string(index=False))

filter_key_columns = {c.split(".")[1] for c in sens_columns}
print(f"\ndistinct filter keys represented inside sensitivity_data: {sorted(filter_key_columns)}")
note_finding("instruments", "sensitivity_data.*",
             f"only {len(carrying)} of {len(inst)} instruments carry any sensitivity_data content, "
             f"and only for a single filter key ('ps1::open')",
             f"{len(carrying)} of {len(inst)} instruments")
note_question(
    "Is ICARE sensitivity metadata sufficiently complete to characterize instrument depth, or does "
    "a complementary source need to be evaluated for this information?",
    f"only {len(carrying)} of {len(inst)} instruments carry any sensitivity_data, covering a single "
    f"filter key",
    "instruments.sensitivity_data.*",
    f"{len(carrying)} of {len(inst)} instruments",
)


sensitivity_data.* columns discovered by Stage 2: ['sensitivity_data.ps1::open.magsys', 'sensitivity_data.ps1::open.zeropoint', 'sensitivity_data.ps1::open.exposure_time', 'sensitivity_data.ps1::open.limiting_magnitude']
                       field  coverage_pct  non_null
            ps1::open.magsys           2.1         2
         ps1::open.zeropoint           2.1         2
     ps1::open.exposure_time           2.1         2
ps1::open.limiting_magnitude           2.1         2

instruments carrying ANY sensitivity_data content: 2 of 95
 id      name    band sensitivity_data.ps1::open.magsys  sensitivity_data.ps1::open.zeropoint  sensitivity_data.ps1::open.exposure_time sensitivity_data.ps1::open.limiting_magnitude
  7 TAROT/TCA optical                                ab                                  28.2                                     180.0                                          19.0
  9 TAROT/TRE optical                                ab                                  2

In [25]:
lim_col = "sensitivity_data.ps1::open.limiting_magnitude"
lim = inst.loc[has_content(inst[lim_col]), lim_col]
print(f"'{lim_col}' python types observed: {sorted({type(v).__name__ for v in lim})}")
print(f"values: {lim.tolist()}")
print(f"stored dtype in the interim table: {inst[lim_col].dtype} (object, because Stage 2 serializes "
      f"mixed int/float scalars as JSON-literal strings to avoid silently coercing one into the other)")

for column in ["sensitivity_data.ps1::open.magsys", "sensitivity_data.ps1::open.zeropoint",
              "sensitivity_data.ps1::open.exposure_time"]:
    present = inst.loc[has_content(inst[column]), column]
    print(f"{column}: {present.tolist()}")


'sensitivity_data.ps1::open.limiting_magnitude' python types observed: ['str']
values: ['19.0', '18']
stored dtype in the interim table: object (object, because Stage 2 serializes mixed int/float scalars as JSON-literal strings to avoid silently coercing one into the other)
sensitivity_data.ps1::open.magsys: ['ab', 'ab']
sensitivity_data.ps1::open.zeropoint: [28.2, 26.3]
sensitivity_data.ps1::open.exposure_time: [180.0, 30.0]


## Allocations — descriptive profile

Identity, the instrument relationship (checked above), `pi`/`group_id`
ownership fields, `types`, `hours_allocated`, and the two sparse list
fields `validity_ranges`/`default_share_group_ids`.


In [26]:
alloc = TABLES["allocations"]
print(f"allocations: {len(alloc)} rows, referencing {alloc['instrument_id'].nunique()} distinct instruments")
print(f"pi coverage: {int(has_content(alloc['pi']).sum())} / {len(alloc)}, "
      f"{alloc['pi'].nunique()} distinct PI names")
print(f"group_id coverage: {int(has_content(alloc['group_id']).sum())} / {len(alloc)}, "
      f"{alloc['group_id'].nunique()} distinct groups")
print(f"proposal_id coverage: {int(has_content(alloc['proposal_id']).sum())} / {len(alloc)}")
if not has_content(alloc["proposal_id"]).any():
    note_finding("allocations", "proposal_id", "entirely empty in this capture", f"0 of {len(alloc)}")

types_lists = alloc["types"].map(json.loads)
print(f"\ntypes: distinct raw combinations = {types_lists.map(tuple).nunique()}")
print(types_lists.map(tuple).value_counts())

print(f"\nhours_allocated coverage: {int(has_content(alloc['hours_allocated']).sum())} / {len(alloc)}")
print(f"hours_allocated: min={alloc['hours_allocated'].min():.6g} / "
      f"median={alloc['hours_allocated'].median():.6g} / max={fmt(alloc['hours_allocated'].max())}")


allocations: 38 rows, referencing 34 distinct instruments
pi coverage: 38 / 38, 26 distinct PI names
group_id coverage: 38 / 38, 3 distinct groups
proposal_id coverage: 0 / 38

types: distinct raw combinations = 7
types
(triggered,)                      23
(triggered, observation_plan)      7
(observation_plan, triggered)      3
(observation_plan,)                2
(triggered, forced_photometry)     1
()                                 1
(forced_photometry,)               1
Name: count, dtype: int64

hours_allocated coverage: 38 / 38
hours_allocated: min=6 / median=100 / max=10,000,000.0


In [27]:
def real_list_content(series):
    def alive(v):
        if v is None:
            return False
        return len(json.loads(v)) > 0
    return series.map(alive)

for column in ["validity_ranges", "default_share_group_ids"]:
    series = alloc[column]
    non_null = int(series.notna().sum())
    non_empty = int(real_list_content(series).sum())
    print(f"{column}: non-null={non_null}, genuinely non-empty list={non_empty} "
          f"(a JSON '[]' does not count as content)")
    if non_empty:
        print(f"  example non-empty value: {series[real_list_content(series)].iloc[0]}")

if not real_list_content(alloc["validity_ranges"]).any():
    note_finding("allocations", "validity_ranges",
                 "every non-null value observed so far is an empty list; genuine content coverage is 0",
                 f"0 of {len(alloc)}")


validity_ranges: non-null=4, genuinely non-empty list=0 (a JSON '[]' does not count as content)
default_share_group_ids: non-null=11, genuinely non-empty list=11 (a JSON '[]' does not count as content)
  example non-empty value: [3,38]


In [28]:
real_time_like_columns = [c for c in alloc.columns if re.search(r"(available|status|state|online|active)", c, re.I)]
print(f"columns in allocations whose name suggests real-time availability: {real_time_like_columns or 'none'}")
print("no explicit real-time availability field was observed in this table")
note_finding("allocations", "<table>",
             "no explicit real-time availability field was observed in this table",
             f"{len(alloc.columns)} columns inspected")
note_question(
    "What does an allocation allow us to claim about resource access, given it carries no explicit "
    "real-time availability field?",
    f"none of allocations' {len(alloc.columns)} columns name an availability/online/status concept; "
    f"hours_allocated ranges from {alloc['hours_allocated'].min():.0f} to {fmt(alloc['hours_allocated'].max())}",
    "allocations.hours_allocated, allocations.types, allocations.validity_ranges",
    f"{len(alloc)} allocations",
)


columns in allocations whose name suggests real-time availability: ['instrument.last_status_update']
no explicit real-time availability field was observed in this table


## Observations — descriptive profile

The 93 historical observation rows: what was actually measured, for
which instrument, in which filter, and with what coverage. No sensitivity
is inferred from these numbers.


In [29]:
obs = TABLES["observations"]
numeric_obs_fields = ["limmag", "exposure_time", "processed_fraction", "airmass", "seeing"]
obs_numeric_rows = []
for field in numeric_obs_fields:
    series = obs[field]
    content = has_content(series)
    present = series[content]
    obs_numeric_rows.append({
        "field": field, "coverage_pct": round(100 * content.sum() / len(obs), 1),
        "non_null": int(content.sum()),
        "min": present.min() if len(present) else None,
        "median": present.median() if len(present) else None,
        "max": present.max() if len(present) else None,
    })
pd.DataFrame(obs_numeric_rows)


,field,coverage_pct,non_null,min,median,max
0,limmag,100.0,93,12.36,16.36,18.0
1,exposure_time,100.0,93,120.00,120.00,180.0
2,processed_fraction,100.0,93,1.00,1.00,1.0
3,airmass,0.0,0,NaN,NaN,NaN
4,seeing,0.0,0,NaN,NaN,NaN


In [30]:
for field in ["airmass", "seeing", "target_name", "field.reference_filters", "field.reference_filter_mags"]:
    coverage = int(has_content(obs[field]).sum())
    if coverage == 0:
        note_finding("observations", field, "entirely empty in this capture", f"0 of {len(obs)}")
    print(f"{field:30s}: {coverage} / {len(obs)} populated")

print(f"\nobstime range: {obs['obstime'].min()} .. {obs['obstime'].max()}")
print(f"exposure_time value counts: {obs['exposure_time'].value_counts().to_dict()}")
print(f"processed_fraction is constant at {obs['processed_fraction'].iloc[0]} across all {len(obs)} rows")


airmass                       : 0 / 93 populated
seeing                        : 0 / 93 populated
target_name                   : 0 / 93 populated
field.reference_filters       : 0 / 93 populated
field.reference_filter_mags   : 0 / 93 populated

obstime range: 2024-05-29T08:10:26.076000 .. 2024-09-21T06:26:02.496000
exposure_time value counts: {120: 86, 180: 7}
processed_fraction is constant at 1.0 across all 93 rows


In [31]:
by_instrument = obs.groupby(["instrument_id", "filt"]).agg(
    n_observations=("id", "size"),
    limmag_min=("limmag", "min"), limmag_median=("limmag", "median"), limmag_max=("limmag", "max"),
    obstime_min=("obstime", "min"), obstime_max=("obstime", "max"),
).reset_index()
by_instrument


,instrument_id,filt,n_observations,limmag_min,limmag_median,limmag_max,obstime_min,obstime_max
0,9,ps1::open,7,18.00,18.000,18.00,2024-07-01T14:41:18.451000,2024-07-01T14:47:18.451000
1,22,bessellr,40,12.36,14.545,15.69,2024-05-29T08:10:26.076000,2024-05-29T09:46:05.081000
2,23,bessellr,46,14.34,17.215,17.54,2024-06-17T02:57:12.758000,2024-09-21T06:26:02.496000


In [32]:
id_match = int((obs["instrument_field_id"] == obs["field.id"]).sum())
alt_match = int((obs["instrument_field_id"] == obs["field.field_id"]).sum())
consistent = int((obs["field.instrument_id"] == obs["instrument_id"]).sum())
print(f"observations.instrument_field_id equals field.id on {id_match} / {len(obs)} rows")
print(f"observations.instrument_field_id equals field.field_id on {alt_match} / {len(obs)} rows")
print(f"observations.field.instrument_id equals observations.instrument_id (self-consistency) on "
      f"{consistent} / {len(obs)} rows")
if id_match == len(obs) and alt_match == 0:
    note_finding("observations", "instrument_field_id / field.id / field.field_id",
                 "instrument_field_id duplicates field.id exactly, while field.field_id is a "
                 "structurally different value; three distinct field-identifier concepts coexist",
                 f"{len(obs)} rows")


observations.instrument_field_id equals field.id on 93 / 93 rows
observations.instrument_field_id equals field.field_id on 0 / 93 rows
observations.field.instrument_id equals observations.instrument_id (self-consistency) on 93 / 93 rows


## Temporal-field inventory

Every column across the four tables whose populated values look like an
ISO-8601 timestamp, found by content rather than by name. This is only a
schema inventory: no conversion or normalization is applied.


In [33]:
ISO_RE = re.compile(r"^\d{4}-\d{2}-\d{2}([T ]\d{2}:\d{2}(:\d{2}(\.\d+)?)?)?$")

def looks_temporal(series, content):
    present = series[content]
    if present.empty or not present.map(lambda v: isinstance(v, str)).all():
        return False
    return present.astype(str).map(lambda v: bool(ISO_RE.match(v.strip('"')))).all()

temporal_rows = []
for table_name in TABLE_NAMES:
    frame = TABLES[table_name]
    for column in frame.columns:
        series = frame[column]
        content = has_content(series)
        if not content.any():
            continue
        if not looks_temporal(series, content):
            continue
        present = series[content].astype(str)
        temporal_rows.append({
            "table": table_name, "column": column, "dtype": str(series.dtype),
            "non_null": int(content.sum()), "min": present.min(), "max": present.max(),
            "representation": "ISO-8601 text",
        })

temporal = pd.DataFrame(temporal_rows).sort_values(["table", "column"]).reset_index(drop=True)
temporal


,table,column,dtype,non_null,min,max,representation
0,allocations,created_at,object,38,2023-01-04T17:44:59.078218,2026-03-11T09:17:18.745104,ISO-8601 text
1,allocations,instrument.created_at,object,38,2022-05-26T13:16:29.168555,2025-08-20T13:53:42.035256,ISO-8601 text
2,allocations,instrument.modified,object,38,2022-05-26T13:16:29.358383,2026-06-07T12:54:04.683091,ISO-8601 text
3,allocations,instrument.telescope.created_at,object,38,2022-05-26T13:16:28.729167,2025-08-20T13:52:58.298542,ISO-8601 text
4,allocations,instrument.telescope.modified,object,38,2022-05-26T13:16:28.729167,2025-08-20T13:52:58.298542,ISO-8601 text
5,allocations,modified,object,38,2023-01-06T08:54:15.779389,2026-06-03T14:56:43.618175,ISO-8601 text
6,instruments,created_at,object,95,2022-05-26T13:16:29.168555,2026-08-05T09:34:18.927608,ISO-8601 text
7,instruments,modified,object,95,2022-05-26T13:16:29.358383,2026-08-05T09:34:18.927608,ISO-8601 text
8,instruments,telescope.created_at,object,95,2022-05-26T13:16:28.729167,2026-08-05T09:31:24.472889,ISO-8601 text
9,instruments,telescope.modified,object,95,2022-05-26T13:16:28.729167,2026-08-05T09:31:24.472889,ISO-8601 text


## Structural families

Groups of flattened columns that share a common nested origin
(`sensitivity_data.*`, `configuration_data.*`, `telescope.*`,
`instrument.*`, `field.*`). Summarizing them as families, rather than as
individual columns, shows how much structure Stage 2's dotted-path
flattening actually produced.


In [34]:
FAMILY_PREFIXES = ["sensitivity_data.", "configuration_data.", "telescope.", "instrument.", "field."]

family_rows = []
for table_name in TABLE_NAMES:
    frame = TABLES[table_name]
    for prefix in FAMILY_PREFIXES:
        family_columns = [c for c in frame.columns if c.startswith(prefix)]
        if not family_columns:
            continue
        any_content = frame[family_columns].apply(has_content).any(axis=1)
        family_rows.append({
            "table": table_name, "family": prefix.rstrip("."),
            "n_columns": len(family_columns),
            "rows_with_any_content": int(any_content.sum()),
            "rows_total": len(frame),
            "structural_variability": "single nested pattern" if len(family_columns) else "n/a",
        })

families = pd.DataFrame(family_rows)
families


,table,family,n_columns,rows_with_any_content,rows_total,structural_variability
0,instruments,sensitivity_data,4,2,95,single nested pattern
1,instruments,configuration_data,1,3,95,single nested pattern
2,instruments,telescope,13,95,95,single nested pattern
3,allocations,instrument,35,38,38,single nested pattern
4,observations,field,9,93,93,single nested pattern


## Cross-table information-coverage summary

One consolidated view of the information categories that appear
potentially relevant to observational-resource characterization, built
only from fields that actually exist in these four tables. This is an
availability map, not a feature list: nothing here is marked required or
final.


In [35]:
coverage_map = [
    {"category": "location", "table": "telescopes", "fields": "lat, lon, elevation",
     "coverage_pct": round(100 * has_content(tel["lat"]).sum() / len(tel), 1), "representation": "direct"},
    {"category": "aperture / diameter", "table": "telescopes", "fields": "diameter",
     "coverage_pct": round(100 * has_content(tel["diameter"]).sum() / len(tel), 1), "representation": "direct"},
    {"category": "operational-mode metadata", "table": "telescopes", "fields": "robotic, fixed_location",
     "coverage_pct": round(100 * has_content(tel["robotic"]).sum() / len(tel), 1), "representation": "direct"},
    {"category": "instrument type", "table": "instruments", "fields": "type",
     "coverage_pct": round(100 * has_content(inst["type"]).sum() / len(inst), 1), "representation": "direct"},
    {"category": "band", "table": "instruments", "fields": "band",
     "coverage_pct": round(100 * has_content(inst["band"]).sum() / len(inst), 1), "representation": "direct"},
    {"category": "filters", "table": "instruments", "fields": "filters",
     "coverage_pct": round(100 * (lengths > 0).sum() / len(inst), 1), "representation": "nested/serialized (JSON list)"},
    {"category": "footprint / region", "table": "instruments", "fields": "region, region_summary, has_region",
     "coverage_pct": round(100 * region_present.sum() / len(inst), 1), "representation": "direct text (DS9 / free text)"},
    {"category": "sensitivity metadata", "table": "instruments", "fields": "sensitivity_data.*",
     "coverage_pct": round(100 * len(carrying) / len(inst), 1), "representation": "nested/serialized"},
    {"category": "allocation / access metadata", "table": "allocations", "fields": "hours_allocated, types, pi, group_id",
     "coverage_pct": round(100 * has_content(alloc["hours_allocated"]).sum() / len(alloc), 1), "representation": "direct"},
    {"category": "historical observation depth", "table": "observations", "fields": "limmag",
     "coverage_pct": round(100 * has_content(obs["limmag"]).sum() / len(obs), 1), "representation": "direct"},
    {"category": "exposure time", "table": "observations", "fields": "exposure_time",
     "coverage_pct": round(100 * has_content(obs["exposure_time"]).sum() / len(obs), 1), "representation": "direct"},
    {"category": "airmass", "table": "observations", "fields": "airmass",
     "coverage_pct": round(100 * has_content(obs["airmass"]).sum() / len(obs), 1), "representation": "direct"},
    {"category": "seeing", "table": "observations", "fields": "seeing",
     "coverage_pct": round(100 * has_content(obs["seeing"]).sum() / len(obs), 1), "representation": "direct"},
]
pd.DataFrame(coverage_map)


,category,table,fields,coverage_pct,representation
0,location,telescopes,"lat, lon, elevation",91.0,direct
1,aperture / diameter,telescopes,diameter,100.0,direct
2,operational-mode metadata,telescopes,"robotic, fixed_location",100.0,direct
3,instrument type,instruments,type,100.0,direct
4,band,instruments,band,100.0,direct
5,filters,instruments,filters,95.8,nested/serialized (JSON list)
6,footprint / region,instruments,"region, region_summary, has_region",20.0,direct text (DS9 / free text)
7,sensitivity metadata,instruments,sensitivity_data.*,2.1,nested/serialized
8,allocation / access metadata,allocations,"hours_allocated, types, pi, group_id",100.0,direct
9,historical observation depth,observations,limmag,100.0,direct


## Data-quality and structural anomaly summary

Every observation flagged with `note_finding(...)` while working through
the sections above, gathered in one place. Each row states what was
measured; none proposes a fix.


In [36]:
quality_summary = pd.DataFrame(QUALITY_FINDINGS)
print(f"{len(quality_summary)} findings recorded")
quality_summary


17 findings recorded


,table,column,observation,measured_scope
0,observations,observation_id,not a unique identifier despite the name,53 distinct values over 93 rows; 'id' is the real primary key
1,telescopes,allocations (nested),telescope id=137 nests allocation id(s) [78] that do not exist in the standalone allocations table (directly or via its instrument),"1 telescope, 1 nested allocation reference"
2,instruments,band,"5 distinct raw labels, including 'Optical'/'optical' case variants and one combined free-text value 'Optical, IR'",90 of 95 instruments carry a case-variant label
3,telescopes,lat/lon/elevation,"8 of 89 telescopes carry no location, all of them non-fixed",8 rows
4,telescopes,morning / evening,"mixed bool/string values: a literal boolean 'false' on some rows, a quoted timestamp-shaped string on others; the literal 'true' was never observed","8 of 89 rows are the boolean form, the rest are the timestamp-shaped string form"
5,instruments,last_status_update,entirely empty in this capture,0 of 95
6,instruments,region / has_region,2 instrument(s) have populated region text while has_region == False,2 of 95 instruments
7,instruments,sensitivity_data.*,"only 2 of 95 instruments carry any sensitivity_data content, and only for a single filter key ('ps1::open')",2 of 95 instruments
8,allocations,proposal_id,entirely empty in this capture,0 of 38
9,allocations,validity_ranges,every non-null value observed so far is an empty list; genuine content coverage is 0,0 of 38


## Questions for the decisions stage

A small set of consequential questions the measurements above justify,
for `B_decisions.ipynb` to resolve. These are questions, not decisions.


In [37]:
questions = pd.DataFrame(DECISION_QUESTIONS)
print(f"{len(questions)} candidate questions recorded")
questions


6 candidate questions recorded


,question,evidence,relevant_table_fields,measured_scope
0,"Historical observations only exercise 3 of 95 instruments — can they serve as evidence of empirical performance for the corpus as a whole, and under what limitations?",observations.instrument_id covers 3 distinct instrument(s) out of 95 instruments and 89 telescopes,"observations.instrument_id, instruments.id","93 observation rows, 3 instruments"
1,Do raw `band`/filter labels require a canonical representation before they can be grouped or compared?,"instruments.band has 5 distinct raw labels for 95 populated rows, 90 of which differ only by case","instruments.band, instruments.filters, observations.filt",90 of 95 instruments
2,How should the mixed `morning`/`evening` values (boolean `false` vs. a timestamp-shaped string) be represented in the final schema?,8 of 89 telescopes carry a literal 'false' in morning/evening while the remaining 81 carry a quoted timestamp-shaped string; 'true' was never observed,"telescopes.morning, telescopes.evening",89 telescopes
3,"Is ICARE's region/footprint information (DS9 text on 19/95 instruments, a free-text region_summary on the same 19) sufficiently complete and structured for the intended telescope-resource layer?","only 19 of 95 instruments carry any footprint information, and it is stored as DS9 region text / free text rather than a structured quantity+unit","instruments.region, instruments.region_summary, instruments.has_region, instruments.number_of_fields",19 of 95 instruments
4,"Is ICARE sensitivity metadata sufficiently complete to characterize instrument depth, or does a complementary source need to be evaluated for this information?","only 2 of 95 instruments carry any sensitivity_data, covering a single filter key",instruments.sensitivity_data.*,2 of 95 instruments
5,"What does an allocation allow us to claim about resource access, given it carries no explicit real-time availability field?","none of allocations' 50 columns name an availability/online/status concept; hours_allocated ranges from 6 to 10,000,000.0","allocations.hours_allocated, allocations.types, allocations.validity_ranges",38 allocations


## Synthesis

**What the four tables provide.** `telescopes` (89 rows) gives identity,
location, aperture and two operational-mode flags whose `morning`/
`evening` pair mixes a literal boolean with a timestamp-shaped string.
`instruments` (95 rows) gives identity, a clean `instrument -> telescope`
link (verified against every row), `type`/`band`/filters, and two
structurally sparse families, `region`/footprint and `sensitivity_data`.
`allocations` (38 rows) gives instrument access, ownership (`pi`,
`group_id`) and `hours_allocated`, but no field that names real-time
availability. `observations` (93 rows) gives 93 empirical measurements —
`limmag`, `exposure_time`, `filt`, `obstime` — concentrated on only 3 of
95 instruments.

**Structural/data-quality findings.** Referential integrity between the
four resources is clean everywhere it was checked: every
`instrument.telescope_id`, `allocation.instrument_id` and
`observation.instrument_id` resolves inside its parent table, and the
nested copies Stage 2 preserved (`telescopes.instruments`,
`instruments.telescope.*`, `allocations.instrument.*`) agree with their
standalone tables on every row but one — a single telescope
(`telescopes.allocations`) nests an allocation id absent from the
standalone `allocations` table. `observations.observation_id` is not a
usable identifier despite its name; `id` is. `instruments.band` carries a
genuine `'Optical'`/`'optical'` case-variant pair on 90 of 95 rows.
`instruments.region`/`has_region` disagree on 2 rows. `airmass`,
`seeing`, `target_name` and `field.reference_filter*` are entirely empty
in this capture.

**Well covered.** Identity and referential-integrity fields; telescope
location/aperture (fixed telescopes only); instrument `type`/`band`/
`filters`; the four historical numeric observation fields that are
populated (`limmag`, `exposure_time`, `processed_fraction`, `obstime`).

**Sparse or ambiguous.** `sensitivity_data.*` (2 of 95 instruments, one
filter key only); `region`/footprint information (19 of 95 instruments,
free text rather than a structured quantity); `configuration_data.*` (3
of 95 instruments); allocation `validity_ranges` (0 of 38 with genuine
content) and `proposal_id` (0 of 38); the 3-instrument concentration of
all historical observations. This sparsity, on its own, is a plausible
motivation to evaluate whether a complementary source is needed later for
sensitivity and footprint information — that evaluation itself belongs to
the decisions stage, not to this notebook.

**Open questions carried into `B_decisions.ipynb`** are listed in full in
the table above; in short: how to represent the mixed `morning`/`evening`
values, whether region/footprint and sensitivity metadata are complete
enough as they stand, whether raw `band`/filter labels need a canonical
form, what an allocation can be taken to mean about access, and whether
the 3-instrument historical observation record can serve as performance
evidence and under what limitation.
